# Anggota 2 — Pemrosesan Pertanyaan

**Tugas Akhir Pemrosesan Bahasa Alami · QA factoid tanpa LLM**

Bagian ini mengubah pertanyaan menjadi **kata kunci, jenis pertanyaan, dan tipe jawaban**.
Pencarian passage dilakukan Anggota 3; ekstraksi jawaban dilakukan Anggota 4.

Hasil: `preprocess.py`, `question_processing.py`, empat ground truth pada
`data/pertanyaan_anggota2.json`, dan keluaran contoh pada folder `output/`.
Buka dengan kernel Python >= 3.10, lalu **Run All**. Modul utama hanya memakai
pustaka standar Python dan tidak memerlukan unduhan model.

**Batas penggunaan:** korpus historis FILKOM 2020, tata tertib 2016, dan penilaian 2015.
`SIAP_RETRIEVAL` berarti bisa diteruskan, bukan bukti bahwa jawaban tersedia.
Implementasi ini disiapkan dengan bantuan AI; mahasiswa perlu memahami, meninjau,
dan menyesuaikannya dengan ketentuan penggunaan AI pada tugas.

## 1. Dasar metode

Modul **PBA14 — Sistem Tanya Jawab**, slide 35–36, membahas *answer type detection*
dan *query formulation*. Slide 43–44 membahas pendekatan berbasis aturan dan regex;
slide 47 membahas kata kunci pencarian. Modul **PBA02 — Pemrosesan Teks**, slide 2,
mencantumkan regex, case folding, tokenization, stemming, dan stopword removal.

Alur: `pertanyaan → normalisasi → deteksi jenis → deteksi tipe jawaban`.
Cabang pencarian: `pertanyaan → tokenisasi + alias → stopword removal → query`.
Kata tanya dikenali sebelum dibuang. Unicode, huruf, spasi, desimal, dan operator
dinormalisasi; singkatan domain disamakan pada kedua sisi.

Stemming, POS tagging, NER, dan dependency parsing **tidak diimplementasikan**.
SIAPA tanpa permintaan nama dipetakan ke JABATAN sebagai asumsi domain, bukan NER.
Satu masukan diharapkan berisi satu pertanyaan eksplisit.

In [1]:
import hashlib
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

cwd = Path.cwd()
kandidat = [cwd, cwd / "Anggota2_Pertanyaan", cwd / "Tugas Akhir" / "Anggota2_Pertanyaan"]
DIR_MODUL = next((p for p in kandidat if (p / "question_processing.py").is_file()), None)
if DIR_MODUL is None:
    raise FileNotFoundError("Jalankan dari folder NLP, Tugas Akhir, atau Anggota2_Pertanyaan.")
DIR_DATA = DIR_MODUL.parent / "Anggota1_Korpus" / "data"
DIR_OUTPUT = DIR_MODUL / "output"
DIR_OUTPUT.mkdir(exist_ok=True)
sys.path.insert(0, str(DIR_MODUL))
from preprocess import VERSI_PREPROCESS, normalisasi_teks, tokenisasi, preprocess, teks_index, preprocess_passage
from question_processing import proses_pertanyaan

def tampil_tabel(baris, kolom):
    def sel(nilai):
        return str(nilai).replace("|", "/").replace("\n", " ")
    isi = ["| " + " | ".join(kolom) + " |", "| " + " | ".join("---" for _ in kolom) + " |"]
    isi += ["| " + " | ".join(sel(b[k]) for k in kolom) + " |" for b in baris]
    display(Markdown("\n".join(isi)))

print("Python:", sys.version.split()[0], "| Versi preprocessing:", VERSI_PREPROCESS)

Python: 3.13.7 | Versi preprocessing: 1.0


## 2. Membaca dan memeriksa korpus

Data Anggota 1 hanya dibaca. Hash input dicatat untuk mendeteksi perubahan
selama eksekusi. ID, metadata dokumen, dan field pencarian diperiksa.

In [2]:
path_passages = DIR_DATA / "passages.jsonl"
path_dokumen = DIR_DATA / "dokumen.json"
hash_awal = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (path_passages, path_dokumen)}
passages = [json.loads(b) for b in path_passages.read_text(encoding="utf-8").splitlines() if b.strip()]
dokumen = json.loads(path_dokumen.read_text(encoding="utf-8"))
passage_by_id = {p["id"]: p for p in passages}
assert passages and len(passage_by_id) == len(passages), "Korpus kosong atau ID duplikat."
for p in passages:
    assert all(isinstance(p[k], str) for k in ("id", "judul_subbab", "konteks", "teks"))
    assert p["teks"].strip() and p["dok_id"] in dokumen
print(f"Terbaca {len(passages)} passage dengan ID unik.")
print("Dokumen:", ", ".join(dokumen))

Terbaca 440 passage dengan ID unik.
Dokumen: pa2020, tatatertib, penilaian


## 3. Tahap preprocessing

`2.00 → 2,00`, `>= → ≥`, dan `<= → ≤`. Token `B+`, negasi `tidak`,
batas `minimal`/`maksimal`, persen, dan angka kata dipertahankan.
Pemisah ribuan dan penyamaan `2,0` dengan `2,00` belum didukung.

Alias: indeks prestasi kumulatif → ipk; indeks prestasi → ip;
satuan/sistem kredit semester → sks; ujian akhir semester → uas;
ujian tengah semester → uts; program studi → prodi.
Dua makna SKS tetap harus dibedakan menggunakan konteks.

In [3]:
contoh_teks = [
    "Berapa IPK minimal untuk lulus?",
    "IPK >= 2.00 dan tidak ada nilai E; kehadiran minimal 80%.",
    "Nilai huruf B+; empat belas semester; 75 < N ≤ 80.",
    "Berapa satuan kredit semester untuk Program Studi?",
]
tampil_tabel([
    {"masukan": t, "token": tokenisasi(t), "token_pencarian": preprocess(t)}
    for t in contoh_teks
], ["masukan", "token", "token_pencarian"])

| masukan | token | token_pencarian |
| --- | --- | --- |
| Berapa IPK minimal untuk lulus? | ['berapa', 'ipk', 'minimal', 'untuk', 'lulus'] | ['ipk', 'minimal', 'lulus'] |
| IPK >= 2.00 dan tidak ada nilai E; kehadiran minimal 80%. | ['ipk', '≥', '2,00', 'dan', 'tidak', 'ada', 'nilai', 'e', 'kehadiran', 'minimal', '80', '%'] | ['ipk', '≥', '2,00', 'tidak', 'ada', 'nilai', 'e', 'kehadiran', 'minimal', '80', '%'] |
| Nilai huruf B+; empat belas semester; 75 < N ≤ 80. | ['nilai', 'huruf', 'b+', 'empat', 'belas', 'semester', '75', '<', 'n', '≤', '80'] | ['nilai', 'huruf', 'b+', 'empat', 'belas', 'semester', '75', '<', 'n', '≤', '80'] |
| Berapa satuan kredit semester untuk Program Studi? | ['berapa', 'sks', 'untuk', 'prodi'] | ['sks', 'prodi'] |

## 4. Jenis pertanyaan, tipe jawaban, dan kata kunci

`jenis_pertanyaan` adalah bentuk tanya, sedangkan `tipe_jawaban` adalah target
ekstraksi untuk Anggota 4. `kata_kunci` tidak berulang; `token_pencarian` dan
`query` mempertahankan urutan dan frekuensi. `aturan_tipe` menunjukkan aturan
yang terpicu, bukan skor keyakinan.

Contoh prioritas: *Berapa bobot nilai huruf B+?* meminta ANGKA.
*Apa nilai huruf untuk nilai angka 78?* meminta NILAI_HURUF.

In [4]:
ground_truth = json.loads((DIR_MODUL / "data" / "pertanyaan_anggota2.json").read_text(encoding="utf-8"))
pertanyaan_contoh = [g["pertanyaan"] for g in ground_truth] + [
    "Berapa bobot nilai huruf B+?",
    "Bagaimana cara menghitung IP?",
    "Apa syarat mengikuti Ujian Skripsi?",
    "Siapa yang menetapkan Dosen Pembimbing Skripsi?",
    "Kapan FILKOM resmi berdiri?",
    "Siapa nama dekan FILKOM sekarang?",
    "Berapa sks maksimal per semester?",
    "Apa?",
]
hasil_contoh = [proses_pertanyaan(q) for q in pertanyaan_contoh]
tampil_tabel(hasil_contoh, ["pertanyaan", "kata_kunci", "jenis_pertanyaan", "tipe_jawaban", "status"])

| pertanyaan | kata_kunci | jenis_pertanyaan | tipe_jawaban | status |
| --- | --- | --- | --- | --- |
| Berapa sks minimal untuk menyelesaikan studi sarjana? | ['sks', 'minimal', 'menyelesaikan', 'studi', 'sarjana'] | BERAPA | ANGKA | SIAP_RETRIEVAL |
| Berapa lama waktu maksimal untuk Ujian Skripsi? | ['lama', 'waktu', 'maksimal', 'ujian', 'skripsi'] | BERAPA | DURASI | SIAP_RETRIEVAL |
| Apa itu skripsi? | ['skripsi'] | APA | DEFINISI | SIAP_RETRIEVAL |
| Apa nilai huruf untuk nilai angka 78? | ['nilai', 'huruf', 'angka', '78'] | APA | NILAI_HURUF | SIAP_RETRIEVAL |
| Berapa bobot nilai huruf B+? | ['bobot', 'nilai', 'huruf', 'b+'] | BERAPA | ANGKA | SIAP_RETRIEVAL |
| Bagaimana cara menghitung IP? | ['cara', 'menghitung', 'ip'] | BAGAIMANA | RUMUS | SIAP_RETRIEVAL |
| Apa syarat mengikuti Ujian Skripsi? | ['syarat', 'mengikuti', 'ujian', 'skripsi'] | APA | SYARAT | SIAP_RETRIEVAL |
| Siapa yang menetapkan Dosen Pembimbing Skripsi? | ['menetapkan', 'dosen', 'pembimbing', 'skripsi'] | SIAPA | JABATAN | SIAP_RETRIEVAL |
| Kapan FILKOM resmi berdiri? | ['filkom', 'resmi', 'berdiri'] | KAPAN | WAKTU | SIAP_RETRIEVAL |
| Siapa nama dekan FILKOM sekarang? | ['nama', 'dekan', 'filkom', 'sekarang'] | SIAPA | ORANG | SIAP_RETRIEVAL |
| Berapa sks maksimal per semester? | ['sks', 'maksimal', 'per', 'semester'] | BERAPA | ANGKA | SIAP_RETRIEVAL |
| Apa? | [] | APA | DEFINISI | PERLU_KLARIFIKASI |

In [5]:
# Ubah untuk mencoba pertanyaan sendiri.
pertanyaan_pengguna = "Berapa IPK minimal untuk menyelesaikan studi sarjana?"
print(json.dumps(proses_pertanyaan(pertanyaan_pengguna), ensure_ascii=False, indent=2))

{
  "pertanyaan": "Berapa IPK minimal untuk menyelesaikan studi sarjana?",
  "normalisasi": "berapa ipk minimal untuk menyelesaikan studi sarjana?",
  "token": [
    "berapa",
    "ipk",
    "minimal",
    "untuk",
    "menyelesaikan",
    "studi",
    "sarjana"
  ],
  "token_pencarian": [
    "ipk",
    "minimal",
    "menyelesaikan",
    "studi",
    "sarjana"
  ],
  "kata_kunci": [
    "ipk",
    "minimal",
    "menyelesaikan",
    "studi",
    "sarjana"
  ],
  "query": "ipk minimal menyelesaikan studi sarjana",
  "jenis_pertanyaan": "BERAPA",
  "penanda_tanya": "berapa",
  "tipe_jawaban": "ANGKA",
  "aturan_tipe": "objek hitungan numerik",
  "status": "SIAP_RETRIEVAL",
  "catatan": [],
  "versi_preprocess": "1.0"
}


## 5. Fungsi yang sama untuk passage

Sesuai SPEK_PASSAGE, teks pencarian adalah `judul_subbab + konteks + teks`.
Contoh pindah program studi menunjukkan pentingnya konteks.
Anggota 3 memakai token ini atau fungsi yang sama, tanpa tokenisasi ulang bawaan
vectorizer yang dapat membuang simbol. Konfigurasi integrasi ada di README.

In [6]:
contoh_passage = passage_by_id["pa2020-III-J-1.1"]
print("ID:", contoh_passage["id"])
print("Judul:", contoh_passage["judul_subbab"])
print("Konteks:", contoh_passage["konteks"])
print("Token pertama:", preprocess_passage(contoh_passage)[:40])
passage_siap = [
    {"passage_id": p["id"], "dok_id": p["dok_id"], "halaman": p["halaman"],
     "token_pencarian": preprocess_passage(p), "versi_preprocess": VERSI_PREPROCESS}
    for p in passages
]
assert all(p["token_pencarian"] for p in passage_siap)
assert all(h["token_pencarian"] == preprocess(h["pertanyaan"]) for h in hasil_contoh)
print(f"Seluruh {len(passage_siap)} passage diproses dengan fungsi bersama.")

ID: pa2020-III-J-1.1
Judul: PERPINDAHAN MAHASISWA
Konteks: 1. Perpindahan Mahasiswa di lingkungan Universitas Brawijaya Merupakan perpindahan mahasiswa antar program studi yang ada di lingkungan UB. Perpindahan antar program studi tersebut terdiri dari: › Adapun persyaratan dan prosedur yang harus dipenuhi dalam pengajuan perpindahan program studi di lingkungan UB antara lain:
Token pertama: ['perpindahan', 'mahasiswa', '1', 'perpindahan', 'mahasiswa', 'lingkungan', 'universitas', 'brawijaya', 'perpindahan', 'mahasiswa', 'antar', 'prodi', 'ada', 'lingkungan', 'ub', 'perpindahan', 'antar', 'prodi', 'terdiri', 'dari', 'adapun', 'persyaratan', 'prosedur', 'harus', 'dipenuhi', 'dalam', 'pengajuan', 'perpindahan', 'prodi', 'lingkungan', 'ub', 'antara', 'lain', '1', 'mahasiswa', 'dapat', 'mengajukan', 'pindah', 'prodi', 'telah']
Seluruh 440 passage diproses dengan fungsi bersama.


## 6. Empat pertanyaan dan bukti jawaban

Gabungkan dengan 16 pertanyaan anggota lain setelah pengecekan duplikasi.
Semua ID dan kutipan diperiksa ke korpus. Konversi `78 → B+` didukung
`75 < 78 ≤ 80`. Ini **jawaban rujukan**, bukan keluaran otomatis chatbot.

In [7]:
sumber_ground_truth = []
for g in ground_truth:
    for pid in g["passage_id"]:
        assert pid in passage_by_id, f"Passage hilang: {pid}"
        p = passage_by_id[pid]
        assert g["bukti"] in p["teks"], f"Kutipan tidak cocok: {pid}"
        sumber_ground_truth.append({
            "no": g["no"], "passage_id": pid,
            "tahun": dokumen[p["dok_id"]]["tahun"], "halaman_PDF": p["halaman"],
        })
    hasil = proses_pertanyaan(g["pertanyaan"])
    assert hasil["tipe_jawaban"] == g["tipe"]
    assert hasil["jenis_pertanyaan"] == g["jenis_pertanyaan"]
assert 75 < 78 <= 80
assert ground_truth[2]["jawaban"].casefold() in passage_by_id["pa2020-VI-A-1"]["teks"].casefold()
tampil_tabel(ground_truth, ["no", "pertanyaan", "jawaban", "tipe"])
tampil_tabel(sumber_ground_truth, ["no", "passage_id", "tahun", "halaman_PDF"])

| no | pertanyaan | jawaban | tipe |
| --- | --- | --- | --- |
| 1 | Berapa sks minimal untuk menyelesaikan studi sarjana? | 144 sks | ANGKA |
| 2 | Berapa lama waktu maksimal untuk Ujian Skripsi? | 75 menit | DURASI |
| 3 | Apa itu skripsi? | Karya ilmiah mahasiswa pendidikan program sarjana yang merupakan wujud dari kajian pengetahuan dan/atau penerapan teknologi berdasarkan kaidah ilmiah dalam bidang studi yang telah dipelajarinya. | DEFINISI |
| 4 | Apa nilai huruf untuk nilai angka 78? | B+ | NILAI_HURUF |

| no | passage_id | tahun | halaman_PDF |
| --- | --- | --- | --- |
| 1 | pa2020-II-I-7 | 2020 | 21 |
| 2 | pa2020-VI-D-10 | 2020 | 43 |
| 3 | pa2020-VI-A-1 | 2020 | 41 |
| 4 | pa2020-tabel3-2 | 2020 | 19 |
| 4 | penilaian-konversi-2 | 2015 | 1 |

## 7. Pemeriksaan aturan dan kasus batas

Kasus buatan berikut adalah **regresi implementasi**, bukan dataset uji
independen atau pengukuran akurasi chatbot. Evaluasi akhir 20 pertanyaan
dan Precision@k/MRR menjadi pekerjaan Anggota 5.

In [8]:
kasus = [
    ("Berapa sks minimal untuk lulus?", "BERAPA", "ANGKA"),
    ("Berapakah IPK minimal untuk lulus?", "BERAPA", "ANGKA"),
    ("Berapa sks maksimal per semester?", "BERAPA", "ANGKA"),
    ("Berapa nilai IPK setelah empat semester?", "BERAPA", "ANGKA"),
    ("Berapa lama batas studi sarjana?", "BERAPA", "DURASI"),
    ("Berapa semester batas studi sarjana?", "BERAPA", "DURASI"),
    ("Berapa maksimal semester masa studi?", "BERAPA", "DURASI"),
    ("Berapa menit durasi Ujian Skripsi?", "BERAPA", "DURASI"),
    ("Berapa persen kehadiran minimal untuk UAS?", "BERAPA", "PERSENTASE"),
    ("Apa itu skripsi?", "APA", "DEFINISI"),
    ("Apa yang dimaksud dengan satuan kredit semester?", "APA", "DEFINISI"),
    ("Apa pengertian indeks prestasi kumulatif?", "APA", "DEFINISI"),
    ("Apa syarat mengikuti Ujian Skripsi?", "APA", "SYARAT"),
    ("Apa saja syarat mengikuti Ujian Skripsi?", "APA", "SYARAT"),
    ("Apa nilai huruf untuk nilai angka 78?", "APA", "NILAI_HURUF"),
    ("Apa bobot nilai huruf B+?", "APA", "ANGKA"),
    ("Apa rentang nilai angka untuk B+?", "APA", "ANGKA"),
    ("Bagaimana cara menghitung IP?", "BAGAIMANA", "RUMUS"),
    ("Apa rumus Indeks Prestasi?", "APA", "RUMUS"),
    ("Bagaimana prosedur cuti akademik?", "BAGAIMANA", "PROSEDUR"),
    ("Siapa yang menetapkan Dosen Pembimbing Skripsi?", "SIAPA", "JABATAN"),
    ("Siapa nama dekan FILKOM sekarang?", "SIAPA", "ORANG"),
    ("Kapan FILKOM resmi berdiri?", "KAPAN", "WAKTU"),
    ("Di mana Praktik Kerja dilaksanakan?", "DI_MANA", "LOKASI"),
    ("Dimana Praktik Kerja dilaksanakan?", "DI_MANA", "LOKASI"),
    ("Mengapa mahasiswa tidak boleh mengikuti UAS?", "MENGAPA", "ALASAN"),
    ("Apakah mahasiswa dengan IPK 2,00 boleh lulus?", "YA_TIDAK", "YA_TIDAK"),
    ("Apa saja jenis sanksi akademik?", "APA", "DAFTAR"),
    ("Halo!", "TIDAK_DIKENALI", "TIDAK_DIKENALI"),
]
for q, jenis, tipe in kasus:
    hasil = proses_pertanyaan(q)
    assert (hasil["jenis_pertanyaan"], hasil["tipe_jawaban"]) == (jenis, tipe), (q, hasil)
assert preprocess("IPK >= 2.00") == preprocess("IPK ≥ 2,00")
assert preprocess("Indeks Prestasi Kumulatif") == preprocess("IPK")
assert preprocess("Ujian Akhir Semester") == preprocess("UAS")
assert preprocess("Satuan Kredit Semester") == preprocess("SKS")
assert {"tidak", "minimal", "maksimal"} <= set(preprocess("tidak minimal maksimal"))
assert {"2,00", "≥", "≤", "80", "%", "b+", "d+"} <= set(preprocess("2,00 ≥ ≤ 80% B+ D+"))
assert preprocess("empat belas semester") == ["empat", "belas", "semester"]
assert proses_pertanyaan("Apa?")["status"] == "PERLU_KLARIFIKASI"
assert proses_pertanyaan("Apa syarat skripsi dan berapa lama ujiannya?")["status"] == "PERLU_KLARIFIKASI"
assert proses_pertanyaan("Siapa nama dekan sekarang?")["catatan"]
assert "prodi" in preprocess_passage(passage_by_id["pa2020-III-J-1.1"])
for kosong in ["", "  ", "\n\t"]:
    try:
        proses_pertanyaan(kosong)
    except ValueError:
        pass
    else:
        raise AssertionError("Masukan kosong harus ditolak.")
try:
    proses_pertanyaan(None)
except TypeError:
    pass
else:
    raise AssertionError("Masukan non-string harus ditolak.")
print(f"LOLOS: {len(kasus)} kasus klasifikasi dan pemeriksaan angka, simbol, konteks, serta masukan batas.")

LOLOS: 29 kasus klasifikasi dan pemeriksaan angka, simbol, konteks, serta masukan batas.


## 8. Menyimpan hasil

Setelah korpus atau preprocessing berubah, jalankan ulang notebook dan bangun
ulang indeks Anggota 3. Teks sumber tetap berada di folder Anggota 1.

In [9]:
path_hasil = DIR_OUTPUT / "hasil_pertanyaan.json"
path_hasil.write_text(json.dumps(hasil_contoh, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
path_token = DIR_OUTPUT / "passages_preprocessed.jsonl"
with path_token.open("w", encoding="utf-8") as berkas:
    for p in passage_siap:
        berkas.write(json.dumps(p, ensure_ascii=False) + "\n")
assert len(path_token.read_text(encoding="utf-8").splitlines()) == len(passages)
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == hash_awal[p.name] for p in (path_passages, path_dokumen))
print(f"Tersimpan {len(hasil_contoh)} contoh pertanyaan dan {len(passage_siap)} passage.")
print("SHA256 input tidak berubah:", json.dumps(hash_awal, indent=2))
print("Keluaran:", path_hasil.name, "dan", path_token.name)

Tersimpan 12 contoh pertanyaan dan 440 passage.
SHA256 input tidak berubah: {
  "passages.jsonl": "204ece8e2db935d28b07b71df752f59677e8d3277f468d8e65c706897a584cfd",
  "dokumen.json": "4515fffeab61a6cd898f9522e7f7fc7a7b6ce88638bb940f64b6dcdd9c5a0798"
}
Keluaran: hasil_pertanyaan.json dan passages_preprocessed.jsonl


## 9. Kesimpulan dan batasan

Hasil pemeriksaan aktual tercetak di atas. Empat ground truth mempunyai bukti
pada korpus, tetapi pengujian ini belum mengukur retrieval atau jawaban chatbot.
Anggota 3 menerima query/token; Anggota 4 menerima tipe jawaban dan teks asli;
Anggota 5 menguji sistem lengkap. Label tipe tambahan masih perlu disepakati.

Regex bergantung susunan kata. Belum ada stemming, koreksi typo, penyamaan semua
sinonim, pemahaman multi-intent, atau bukti answerability. Angka kata hanya
dipertahankan; konversi dan ekstraksinya dikerjakan Anggota 4.
Modul jawaban tetap harus menangani informasi yang tidak ditemukan.

### Rujukan

Adikara, P.P., Setiawan, B.D. and Adinugroho, S. (n.d.) *Pemrosesan Teks*.
Bahan kuliah Pemrosesan Bahasa Alami, Universitas Brawijaya, slide 2.

Adikara, P.P., Setiawan, B.D. and Adinugroho, S. (n.d.) *Sistem Tanya Jawab
(Question Answer System)*. Bahan kuliah Pemrosesan Bahasa Alami,
Universitas Brawijaya, slide 35–36 dan 43–47.

Jurafsky, D. and Martin, J.H. (2021) 'Question Answering', in *Speech and Language
Processing*. 3rd edn draft, 29 December, Chapter 23. Available at:
https://web.stanford.edu/~jurafsky/slp3/old_dec21/23.pdf
(Accessed: 22 November 2026).

Tahun tidak terlihat pada sampul dua modul, sehingga digunakan n.d.
Buku menjelaskan konsep pemrosesan pertanyaan dan tipe jawaban;
daftar aturan Indonesia di sini adalah rancangan implementasi untuk korpus ini.